<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP project

In [1]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00


In [24]:
#DRIVE
from google.colab import drive
drive.mount('/content//NLP_proj')

Mounted at /content//NLP_proj


In [2]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch


In [3]:
# Load the model
VERBOSE = True
CHOSEN = 'llama'
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"},
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf"},
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

In [4]:
rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/dominion.txt').text
rulebook[:100]

'# Dominion\nYou are a monarch, like your parents before you - a ruler of a small pleasant kingdom of '

In [5]:
# Check that input is inside context window
#tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

#tokens = tokenizer.encode(rulebook)
#print(len(tokens))

In [6]:
def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

In [7]:
import gc
def multiple_model_test(prompts, file_names, iterations, test_name):
    outputs = defaultdict(dict)

    for game,prompt_name,prompt,it in tqdm([(f,pn,p,it) for f in file_names for (pn,p) in prompts for it in range(iterations)]):
        rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
        out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)
        outputs[CHOSEN+'-'+game+'-'+prompt_name][str(it)] = out['choices'][0]['message']['content']

    with open(f'{test_name}.out','w') as f:
        json.dump(dict(outputs),f)

    return outputs

# Rule extraction
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [8]:
prompts = [("kid", """You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation"""),
           ("analogies", """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g. some other famous board games).
           se the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""),
]
game_names = [ 'dominion','7_wonders', 'catan', 'power_grid_recharged','ticket_to_ride',]

iterations = 1

multiple_model_test(prompts, game_names, iterations, 'extraction')

100%|██████████| 10/10 [03:58<00:00, 23.84s/it]


defaultdict(dict,
            {'llama-dominion-kid': {'0': "Hey there, young adventurer! Let's talk about the game Dominion. \n\n**What's the goal of the game?**\nThe goal of the game is to build a deck of cards that will help you win the most points. You'll be collecting cards that will give you money, help you buy new cards, and even defend against other players.\n\n**How do you win the game?**\nYou win the game by having the most points at the end. Points are earned by collecting Victory cards, which are worth points. The player with the most points wins the game!\n\n**What happens during a turn?**\nA turn has three phases: Action, Buy, and Clean-up.\n\n1. **Action phase**: You can play one Action card from your hand. Action cards do cool things like give you more cards, money, or even defend against other players.\n2. **Buy phase**: You can play Treasure cards to earn money, and then use that money to buy new cards from the Supply.\n3. **Clean-up phase**: You put all the cards you 

## Error detection

 Each rulebook is edited by inserting a set of 5 errors each of increasing difficulty:
 - level 0: **original** -> unaltered rulebook
 - level 1: **missing** -> an entire paragraph of the rulebook describing some core mechanic is missing
- level 2: **unsolvable** -> (in one line states you can draw two cards, in another one that you can draw only one)
- level 3: **incoherent** -> a mechanic that hardlocks the game (you cannot play train if you do not have train on the map but on another line clearly states you start with an empty map)
- level 4: **gamebreaking** -> a coherent but obviously unbalanced mechanic

In [19]:
prompts = ["""You are an expert game board player.
Examine the rulebook provided by the user.
Proceed with a chain of thought:

- Scan the text linearly and note any statements that conflict with earlier ones.
- For each of the game mechanic check if it is explained.
- Check whether any mechanic could halt the game or give a player an overwhelming advantage.

Report the **first** problem you discover, quoting the relevant line and summarizing its impact.
If nothing stands out, reply “The rules appear consistent.”
"""]
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 5


output_dict = {str(g):{'lvl'+str(l): {str(it): '' for it in range(iterations)} for l in range(5)} for g  in game_names}


for game,prompt,it,lvl in tqdm([(f,p,it,lvl) for f in file_names for p in prompts for it in range(iterations) for lvl in range(5)]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/lvl'+str(lvl)+'/'+game+'.txt').text
    out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)['choices'][0]['message']['content']
    if(VERBOSE):
        print(out)
    output_dict[game]['lvl'+str(lvl)][str(it)] = out

with open(f'{CHOSEN}_error_detection.out','w') as f:
    json.dump(dict(output_dict),f)

  1%|          | 1/100 [00:09<16:11,  9.81s/it]

The first problem I discover is that the rules for drawing face-up locomotives are not entirely clear.

Relevant line: "If, at any time, 3 of the 5 face up train cards are locomotives, all 5 cards are immediately discarded, and 5 new cards are turned face up to replace them. If you want to draw a face up locomotive, it must be the first train card you draw this turn and you cannot then draw a second card."

Impact: This rule creates a conflict between two different situations:

1. If a player wants to draw a locomotive as their first card, they can do so, but then they cannot draw a second card.
2. However, if a player draws a locomotive as their second card (i.e., after drawing a non-locomotive card first), it is not clear whether they are allowed to do so.

This ambiguity could lead to disagreements or disputes during gameplay. A clearer rule would be needed to resolve this situation.


  2%|▏         | 2/100 [00:18<15:07,  9.26s/it]

The first problem I discover is that the rulebook does not clearly explain what happens when a player draws a locomotive card as their first train card, and then draws a second card from the face-up pile. The relevant line is:

"If, at any time, 3 of the 5 face up train cards are locomotives, all 5 cards are immediately discarded, and 5 new cards are turned face up to replace them."

However, this rule is not applied consistently throughout the game. In the "Draw Train Cards" section, it is stated that "If, at any time, 3 of the 5 face up train cards are locomotives, all 5 cards are immediately discarded, and 5 new cards are turned face up to replace them." This implies that this rule applies during the player's turn, but it is not clear how this rule affects the player's ability to draw a second card after drawing a locomotive as their first card.

This ambiguity could lead to disputes during gameplay, and it is unclear how this situation would be resolved.


  3%|▎         | 3/100 [00:27<14:45,  9.13s/it]

...

The rules appear consistent.


 99%|█████████▉| 99/100 [22:40<00:14, 14.30s/it]

The first problem I discover is a conflict in the rules regarding the "discount token" and the power plant market.

In Phase 2: Auction Power Plants, the rules state that the discount token is placed on the smallest power plant in the current market, and the minimum bid for this power plant is reduced to 1 Elektro regardless of the actual number of the power plant. However, in Phase 5: Bureaucracy, the rules state that when the Step 3 card is drawn, it is treated as the highest power plant for the remainder of the phase, and it is placed at the end of the future market. This implies that the discount token is removed from the smallest power plant in the current market.

The problem arises when the Step 3 card is drawn during Phase 2, and the discount token is removed from the smallest power plant in the current market, but the power plant is still considered the smallest in the market due to the discount token being placed on it earlier. This creates a conflict, as the rules are not cl

100%|██████████| 100/100 [22:58<00:00, 13.78s/it]

The first problem I discover is a conflict in the rules regarding the power plant market.

In the section "The Power Plants", it is stated that "When adding new power plants to the power plant market during the game, always rearrange all power plants in the power plant market in ascending order with the 4 cheapest plants in the current market."

However, in the section "Phase 2: Auction Power Plants", it is stated that "Immediately, draw a new power plant from the power plant stack to replace the one bought and place it in the power plant market. Rearrange the power plants in ascending order of numbers: the four lowest in the current market, and the higher power plants in the future market."

This creates a conflict because the first statement implies that new power plants should be added to the current market, while the second statement implies that new power plants should be added to the future market.

This conflict could lead to inconsistencies in gameplay and potentially allow a p

In [11]:
output_dict

{'dominion': {'lvl0': {'0': 'After carefully examining the rulebook, I have identified a potential game-breaking problem:\n\n**The rules do not specify how to handle the case where a player\'s hand is empty and they need to draw a card, but there are no cards left in the deck.**\n\nIn the "Clean-up Phase" section, it is stated that "If your deck has fewer than 5 cards, first shuffle your discard pile and put it under your deck, then draw." This implies that if a player\'s deck is empty, they should shuffle their discard pile and use it as their new deck. However, there is no clear guidance on what happens if the discard pile is also empty.\n\nThis could lead to a situation where a player is unable to draw a card, and the game becomes stuck. To resolve this, the rules might need to be clarified to specify what happens in this scenario, such as allowing the player to draw from the Supply or using a randomizer mechanism to add new cards to the game.\n\nThis is not an immediately obvious p

In [20]:
# EVALUATION
sys_prompt = """You are a rule‑checking assistant.
Read ANSWER and GROUND TRUTH, narrate whether the explanations refer to the same rule, and conclude with a single word **CORRECT** or **WRONG** on the final line.

### Examples
input:'''
ANSWER -> The Monopoly win condition states that the game ends when all other players have gone bankrupt,leaving a single player with all the assets. However, this rule can be problematic because it assumes that players will continue to take turns until bankruptcy occurs, which may never happen if a player repeatedly lands on “Free Parking” and collects cash without paying any fees. In such a scenario, the game could drag on indefinitely, making the win condition effectively unreachable. A more practical rule might impose a turn limit or a cash‑threshold to declare a winner when the game stalls.
GROUND TRUTH-> The problem is that the first player may reroll the dice any time.
'''

output:'''
The answer does not talk about rerolling the dice
WRONG
'''

---

input:'''
ANSWER ->  After examining the rulebook, I've identified a **gamebreaking** problem: In the first phase of the game rules states that when a player draws a card any other player can choose to draw another card. This could create a scenario where all player are willing to draw until the deck is empty.
GROUND TRUTH -> infinite card draw at the start of the game.
'''
output:'''
the answer and the ground truth adress the card draw in first phases of the game as the problem
CORRECT
'''

---

input:'''
ANSWER -> The rules appear consistent.
GROUND TRUTH -> The rulebook is consistent.
'''
'''
output:'''
both statements refer to the rule being consistent
CORRECT
'''
"""
ground_truth = {
    "ticket_to_ride": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "In the rulebook it is not described how to claim a route",
        "lvl2": "In one section it is stated that during its turn a player may draw up to 2 train cards but on another line states that a player may draw up to 7 cards",
        "lvl3": "The \"a player cannot claim a route unless they have already claimed at least one other contiguous route earlier in the game\" creates a situation where a player may not clame its first route since it is not contiguous to other routes",
        "lvl4": "The rule that allows a player to reveal cards until they reveal a locomotive is unbalanced since it always allow a player to draw 2 locomotives per turn"
    },
    "dominion": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "the buy phase section is missing",
        "lvl2": "the sentence \"After buying a card, a player may **continue to play additional Treasure cards** from their hand to increase their total  for that turn\" this is in direct contrast with a previous statement that does not allow a player to play treasures after the buy phase",
        "lvl3": "the sentence \"You cannot play cards that grant more action as the first action of the turn\" creates a situation where a player can't play card that grant extra action.",
        "lvl4": "the ability to play treasures also from the discard pile is unbalanced since allows player to have an ever increasing amount of gold to spend during their turn removing one of the balancing aspect of the game"
    },
    "7_wonders": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "missing section explaining the resolution of military conflicts phase",
        "lvl2": "the sentence \"When buying a resource you may use Coins received from a neighbor earlier in the same turn to pay the 2‑Coin cost.\" contradicts a previous rules that states that a player cannot use coin recieved during the same turn to pay for a resource",
        "lvl3": "infinite resolve conflict loop",
        "lvl4": "the rule that allow a player to construct one stage of your Wonder for free is very unbalanced"
    },
    "catan": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "missing subsection explaining building rules",
        "lvl2": "the sentence \"When you roll a 7, all hexes produce a resource except for the one with the robber.\" is directly in contrast with the rule stating that \"When you roll a 7, hexes do not produce any resources\"",
        "lvl3": "rules state that You can only trade a resource produced this turn by the hex with the robber, however since the hex with the robber does not produce resources this means that trading is not possible",
        "lvl4": "the rule stating that robber steals all resources from all players is very unbalanced since the player rolling a 7 is very likely to win the game"
    },
    "power_grid_recharged": {
        "lvl0": "the rulebook is consistent",
        "lvl1": "missing some steps in phase 5",
        "lvl2": "the sentence \"As explained in the section “The Power Plants”, each power plant may store only the number of resource tokens matching the number of symbols on the card and needs twice that number of tokens to produce electricity\" is contradicted later in the rules",
        "lvl3": "There is no way to reach phase 3 since the \"step 3\" card ir removed at the start of the game",
        "lvl4": "the rule granting a player who does not supply any city the same amount of Elektro as the highest‑earning player is unbalanced since it means that the player spending less resources is also gaining more Elektro than the others"
    }
}

verbose = True


for lvl in range(5):
    res = []
    # construct the answer pair
    answer_pair = []
    for game in game_names:
        for x in output_dict[game][f'lvl{lvl}']:
            answer_pair.append((x, ground_truth[game][f'lvl{lvl}']))
    # check
    for answer, real in answer_pair:
        usr_prompt = f"ANSWER -> {answer}\nGROUND TRUTH -> {real}"
        out = model.create_chat_completion(generate_message(sys_prompt, usr_prompt), temperature=0.7)
        if(verbose):
            print(out['choices'][0]['message']['content'])
        res.append(out['choices'][0]['message']['content'].splitlines()[-1])
    print(f"lvl{lvl}: {res.count('CORRECT')}/{len(res)}")


both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
The answer and the ground truth both refer to the rulebook being consistent.
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
Both statements refer to the rule being consistent
CORRECT
Both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the rule being consistent
CORRECT
both statements refer to the r

# Game classification
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

## Estimating everything at once

In [25]:
prompts = ["""You are a board‑game analyst that always explains its reasoning before answering.
For any supplied rule excerpt you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting BGG mechanic(s).
3. Judge the rule density and decision depth. Then assign a complexity score (1‑5).
4. From the number of components in the box and player‑interaction patterns infer the optimal player‑count range.
6. Assess the progression of a typical game turn. Based on the complexity of the required actions and how much each turn brings the player closer to the final goal, estimate the game’s average duration in minutes.
Finally, output a **single CSV line**:
<game_name>,<game_class>,<game_complexity>,<minimum_players>, <maximum_players>, <duration>
"""]
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 10
res = []

for game,prompt,it,lvl in tqdm([(f,p,it,lvl) for f in file_names for p in prompts for it in range(iterations) for lvl in range(5)]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/lvl'+str(lvl)+'/'+game+'.txt').text
    out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)['choices'][0]['message']['content']
    if(VERBOSE):
        print(out)
    res.append(out['choices'][0]['message']['content'].splitlines()[-1].split(','))

with open(f'{CHOSEN}_estimation.out','w') as f:
    json.dump(dict(output_dict),f)


  0%|          | 0/200 [00:16<?, ?it/s]

**Key Actions and Components:**

1. Train cards: used to claim routes
2. Scoring markers: track players' scores
3. Tickets: secretly kept by players, completed for points
4. Plastic trains: used to claim routes, form longest path
5. Longest path bonus card: awarded to player with longest path
6. Route claiming: players claim routes on the board
7. Drawing train cards: players draw train cards to claim routes
8. Drawing tickets: players draw tickets to complete
9. Game end: players calculate final scores, longest path is determined

**BGG Mechanics:**

1. Route Building (e.g., Ticket to Ride, Rails of New England)
2. Hand Management (e.g., drawing train cards, ticket drawing)
3. Path Building (e.g., creating longest path)
4. Variable Player Powers (e.g., secretly kept tickets)

**Rule Density and Decision Depth:**

The rules are relatively dense, with many exceptions and nuances (e.g., locomotive cards, double-routes, longest path calculation). The decision depth is moderate, as players

TypeError: string indices must be integers, not 'str'

## Estimating each parameter separately

In [ ]:
prompt = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting BGG mechanic(s).

here is a complete list of BGG mechanics:
Acting, Action / Event, Action Drafting, Action Points, Action Queue, Action Retrieval,
Action Timer, Advantage Token, Alliances, Area Majority / Influence, Area Movement, Area-Impulse,
Auction / Bidding, Auction Compensation, Auction: Dexterity, Auction: Dutch, Auction: Dutch Priority,
Auction: English, Auction: Fixed Placement, Auction: Multiple Lot, Auction: Once Around,
Auction: Sealed Bid,Auction: Turn Order Until Pass, Automatic Resource Growth, Betting and Bluffing,
Bias, Bids As Wagers, Bingo, Bribery, Campaign / Battle Card Driven, Card Play Conflict Resolution,
Catch the Leader, Chaining, Chit-Pull System, Closed Drafting, Closed Economy Auction, Command Cards,
Commodity Speculation, Communication Limits, Connections, Constrained Bidding, Contracts,
Cooperative Game, Crayon Rail System, Critical Hits and Failures, Cube Tower, Deck Construction, "Deck, Bag, and Pool Building", Deduction,Delayed Purchase, Dice Rolling, Die Icon Resolution, Different Dice Movement, Drawing, Elapsed Real Time Ending, Enclosure, End Game Bonuses, Events, Finale Ending, Flicking, Follow, Force Commitment, Grid Coverage, Grid Movement, Hand Management, Hexagon Grid, Hidden Movement, Hidden Roles, Hidden Victory Points, Highest-Lowest Scoring, Hot Potato, "I Cut, You Choose", Impulse Movement, Income, Increase Value of Unchosen Resources, Induction, Interrupts, Investment, Kill Steal, King of the Hill, Ladder Climbing, Layering, Legacy Game, Line Drawing, Line of Sight, Loans, Lose a Turn, Mancala,
Map Addition, Map Deformation, Map Reduction, Market, Matching, Measurement Movement,
Melding and Splaying, Memory, Minimap Resolution, Modular Board, Move Through Deck,
Movement Points, Movement Template, Moving Multiple Units, Multi-Use Cards, Multiple Maps,
Narrative Choice / Paragraph, Negotiation, Neighbor Scope, Network and Route Building,
Once-Per-Game Abilities, Open Drafting, Order Counters, Ordering, Ownership, Paper-and-Pencil,
Passed Action Token, Pattern Building, Pattern Movement, Pattern Recognition, Physical Removal,
Pick-up and Deliver, Pieces as Map, Player Elimination, Player Judge, Point to Point Movement,
Predictive Bid, Prisoner's Dilemma, Programmed Movement, Push Your Luck, Questions and Answers, Race,
Random Production, Ratio / Combat Results Table, Re-rolling and Locking, Real-Time, Relative Movement,
Resource Queue, Resource to Move, Rock-Paper-Scissors, Role Playing, Roles with Asymmetric Information,
Roll / Spin and Move, Rondel, Scenario / Mission / Campaign Game, Score-and-Reset Game, Secret Unit Deployment,
Selection Order Bid, Semi-Cooperative Game, Set Collection, Simulation, Simultaneous Action Selection,
Singing, Single Loser Game, Slide / Push, Solo / Solitaire Game, Speed Matching, Spelling, Square Grid,
Stacking and Balancing, Stat Check Resolution, Static Capture, Stock Holding, Storytelling, Sudden Death Ending,
Tags, Take That, Targeted Clues, Team-Based Game, Tech Trees / Tech Tracks, Three Dimensional Movement,
Tile Placement, Track Movement, Trading, Traitor Game, Trick-taking, Tug of War, Turn Order: Auction,
Turn Order: Claim Action, Turn Order: Pass Order, Turn Order: Progressive, Turn Order: Random,
Turn Order: Role Order, Turn Order: Stat-Based, Turn Order: Time Track, Variable Phase Order,
Variable Player Powers, Variable Set-up, Victory Points as a Resource, Voting, Worker Placement,
Worker Placement with Dice Workers, "Worker Placement, Different Worker Types", Zone of Control
"""
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 10
res = []

for game,prompt,it,lvl in tqdm([(f,p,it,lvl) for f in file_names for p in prompts for it in range(iterations) for lvl in range(5)]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/flawed_texts/lvl'+str(lvl)+'/'+game+'.txt').text
    out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)['choices'][0]['message']['content']
    if(VERBOSE):
        print(out)
    res.append(out['choices'][0]['message']['content'].splitlines()[-1].split(','))

with open(f'{CHOSEN}_estimation.out','w') as f:
    json.dump(dict(output_dict),f)


In [ ]:
%pip install bgg-api

In [ ]:
# EVALUATION
from boardgamegeek import BGGClient
bgg = BGGClient()
bgg_gt = []

for gn in ["Ticket to Ride", "Dominion", "Catan", "Power Grid Recharged"]:
    game = bgg.game(gn)
    bgg_gt.append(gn.lower().replace(" ", "_"), game.mechanics, game.complexity, game.minplayers, game.maxplayers, game.playingtime)
print(bgg_gt)

